# Day 27: ReAct Prompting & Reasoning – From Scratch

In [ ]:
import os
import re
import wikipedia
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## 1. Define tools (calculator, Wikipedia search)

In [ ]:
def calculator(expression: str) -> str:
    try:
        result = eval(expression)
        return f"Observation: {result}"
    except Exception as e:
        return f"Observation: Error - {e}"

def wikipedia_search(query: str) -> str:
    try:
        summary = wikipedia.summary(query, sentences=2)
        return f"Observation: {summary}"
    except wikipedia.exceptions.DisambiguationError as e:
        return f"Observation: Ambiguous, try one of: {e.options[:3]}"
    except Exception as e:
        return f"Observation: Not found"

## 2. ReAct prompt template

In [ ]:
react_prompt = """You are a ReAct agent. Answer the question by alternating Thought and Action.

Available actions:
- Calculate[expression]: evaluate math
- Search[query]: search Wikipedia
- Finish[answer]: provide final answer

Question: {question}

{history}
Thought:"""

## 3. Parse LLM output to extract action

In [ ]:
def parse_action(text):
    # Look for patterns like Calculate[...] or Search[...] or Finish[...]
    calculate_match = re.search(r'Calculate\[(.*?)\]', text)
    if calculate_match:
        return ("calculate", calculate_match.group(1))
    search_match = re.search(r'Search\[(.*?)\]', text)
    if search_match:
        return ("search", search_match.group(1))
    finish_match = re.search(r'Finish\[(.*?)\]', text)
    if finish_match:
        return ("finish", finish_match.group(1))
    return ("unknown", None)

## 4. ReAct loop

In [ ]:
def react_agent(question, max_steps=5):
    history = ""
    for step in range(max_steps):
        prompt = react_prompt.format(question=question, history=history)
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        output = response.choices[0].message.content
        print(f"Step {step+1}:\n{output}\n")
        
        action, arg = parse_action(output)
        if action == "finish":
            return arg
        elif action == "calculate":
            observation = calculator(arg)
        elif action == "search":
            observation = wikipedia_search(arg)
        else:
            observation = "Observation: Invalid action, try again."
        
        history += output + "\n" + observation + "\n"
    return "Max steps reached without finish."

answer = react_agent("What is the population of France divided by 2?")
print("Final answer:", answer)